# WorldSim DGX Spark Generation Analyzer

Analyze a `sample_generations.jsonl` artifact directly in JupyterLab without using the CLI manually.


## Environment / Path Check


In [1]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
{
    "python_executable": sys.executable,
    "cwd": os.getcwd(),
    "repo_root": str(REPO_ROOT.resolve()),
}


{'python_executable': '/home/hyunlord/github/diffusion-study/.venv/bin/python3',
 'cwd': '/home/hyunlord/github/worldsim-training',
 'repo_root': '/home/hyunlord/github/worldsim-training'}

## Config

Edit this cell first.


In [2]:
from pathlib import Path

ARTIFACT_DIR = Path("outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z")
SAMPLE_FILE = ARTIFACT_DIR / "sample_generations.jsonl"
SUMMARY_FILE = ARTIFACT_DIR / "summary.json"
RUN_CONFIG_FILE = ARTIFACT_DIR / "run_config.json"
REPORT_OUTPUT_FILE = ARTIFACT_DIR / "analysis_report.json"
EXAMPLES_PER_CATEGORY = 3

{
    "ARTIFACT_DIR": str(ARTIFACT_DIR),
    "SAMPLE_FILE": str(SAMPLE_FILE),
    "SUMMARY_FILE": str(SUMMARY_FILE),
    "RUN_CONFIG_FILE": str(RUN_CONFIG_FILE),
    "REPORT_OUTPUT_FILE": str(REPORT_OUTPUT_FILE),
    "EXAMPLES_PER_CATEGORY": EXAMPLES_PER_CATEGORY,
}


{'ARTIFACT_DIR': 'outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z',
 'SAMPLE_FILE': 'outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z/sample_generations.jsonl',
 'SUMMARY_FILE': 'outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z/summary.json',
 'RUN_CONFIG_FILE': 'outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z/run_config.json',
 'REPORT_OUTPUT_FILE': 'outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z/analysis_report.json',
 'EXAMPLES_PER_CATEGORY': 3}

## File Existence Check


In [3]:
required_paths = {
    "sample_file": SAMPLE_FILE,
    "summary_file": SUMMARY_FILE,
    "run_config_file": RUN_CONFIG_FILE,
}
resolved_paths = {name: str(path.resolve()) for name, path in required_paths.items()}
missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required artifacts: {', ' .join(missing)}")
resolved_paths


{'sample_file': '/home/hyunlord/github/worldsim-training/outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z/sample_generations.jsonl',
 'summary_file': '/home/hyunlord/github/worldsim-training/outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z/summary.json',
 'run_config_file': '/home/hyunlord/github/worldsim-training/outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z/run_config.json'}

## Load Artifact Preview


In [4]:
import json

from tools.generation_analyzer import load_samples

samples = load_samples(SAMPLE_FILE)
summary_payload = json.loads(SUMMARY_FILE.read_text(encoding="utf-8"))
run_config_payload = json.loads(RUN_CONFIG_FILE.read_text(encoding="utf-8"))

{
    "sample_preview": [
        {
            "task": row.get("task"),
            "generated_assistant": str(row.get("generated_assistant", ""))[:300],
            "raw_generated_assistant": str(row.get("raw_generated_assistant", ""))[:300],
        }
        for row in samples[:3]
    ],
    "summary": summary_payload,
    "run_config": run_config_payload,
}


{'sample_preview': [{'task': 'A',
   'generated_assistant': '{"text_ko": "다귀질", "text_en": "sanguine", "register": "haera", "temperament_expressed": "haera", "dominant_trait": "hao", "stressed": 0.3}',
   'raw_generated_assistant': '{"text_ko": "다귀질", "text_en": "sanguine", "register": "haera", "temperament_expressed": "haera", "dominant_trait": "hao", "stressed": 0.3}'},
  {'task': 'B',
   'generated_assistant': '{"text_ko": "과제", "text_en": "task", "register": "haera", "emotion_expressed": ["joy", "sadness", "fear", "anger", "trust", "disgust", "surprise", "anticipation"], "temperament_influence": ["melancholic", "melancholic_rhyme", "melancholic_sense", "melancholic_mood", "melancholic_tone", "melancholic_',
   'raw_generated_assistant': '{"text_ko": "과제", "text_en": "task", "register": "haera", "emotion_expressed": ["joy", "sadness", "fear", "anger", "trust", "disgust", "surprise", "anticipation"], "temperament_influence": ["melancholic", "melancholic_rhyme", "melancholic_sense", "

## Run Analyzer


In [5]:
import json

from tools.generation_analyzer import generate_report, recommend_next_action

analysis_report = generate_report(samples, examples_per_category=EXAMPLES_PER_CATEGORY)
REPORT_OUTPUT_FILE.write_text(json.dumps(analysis_report, ensure_ascii=False, indent=2), encoding="utf-8")
recommendation = recommend_next_action(analysis_report)

{
    "report_output_file": str(REPORT_OUTPUT_FILE.resolve()),
    "overall_status": analysis_report.get("overall_status"),
    "recommended_next_action": recommendation,
}


{'report_output_file': '/home/hyunlord/github/worldsim-training/outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z/analysis_report.json',
 'overall_status': 'prompt_leakage_issue',
 'recommended_next_action': {'status': 'enum_instability',
  'recommended_next_action': 'Stabilize task-specific enum generation before the next smoke run.'}}

## Analysis Summary


In [6]:
{
    "total_samples": analysis_report.get("total_samples"),
    "counts_by_primary_category": analysis_report.get("counts_by_failure_category"),
    "malformed_json_count": analysis_report.get("malformed_json_count"),
    "truncation_count": analysis_report.get("truncation_count"),
    "fenced_json_count": analysis_report.get("fenced_json_count"),
    "trailing_text_count": analysis_report.get("trailing_text_count"),
    "enum_drift_count": analysis_report.get("enum_drift_count"),
    "language_drift_count": analysis_report.get("language_drift_count"),
    "semantic_low_quality_count": analysis_report.get("semantic_low_quality_count"),
    "semantic_drift_count": analysis_report.get("semantic_drift_count"),
    "affected_tasks_summary": analysis_report.get("affected_tasks_summary"),
    "overall_status": analysis_report.get("overall_status"),
}


{'total_samples': 7,
 'counts_by_primary_category': {'enum_drift': 1, 'ok': 4, 'prompt_leakage': 2},
 'malformed_json_count': 0,
 'truncation_count': 0,
 'fenced_json_count': 0,
 'trailing_text_count': 0,
 'enum_drift_count': 1,
 'language_drift_count': 0,
 'semantic_low_quality_count': 0,
 'semantic_drift_count': 0,
 'affected_tasks_summary': {'B': {'enum_drift': 1},
  'C': {'prompt_leakage': 1},
  'G': {'prompt_leakage': 1}},
 'overall_status': 'prompt_leakage_issue'}

## Example Failures


In [7]:
analysis_report.get("example_failures", {})


{'enum_drift': [{'task': 'B',
   'generated_assistant': '{"text_ko": "과제", "text_en": "task", "register": "haera", "emotion_expressed": ["joy", "sadness", "fear", "anger", "trust", "disgust", "surprise", "anticipation"], "temperament_influence": ["melancholic", "melancholic_rhyme", "melancholic_sense", "melancholic_mood", "melancholic_tone", "melancholic_vocalization", "melancholic_phoneme", "melancholic_accent"], "stress_level": 0.4}',
   'json_failure': {'raw_parseable_json': True,
    'fence_stripped_parseable_json': True,
    'fenced_json': False,
    'raw_json_parse_error': None,
    'stripped_json_parse_error': None,
    'category': 'ok'},
   'prompt_leakage': None,
   'enum_drift': [{'field_name': 'emotion_expressed',
     'expected_enum_set': ['anger',
      'anticipation',
      'disgust',
      'fear',
      'joy',
      'sadness',
      'surprise',
      'trust'],
     'actual_value': '["joy", "sadness", "fear", "anger", "trust", "disgust", "surprise", "anticipation"]',
    

## Recommended Next Action


In [8]:
{
    "overall_status": analysis_report.get("overall_status"),
    "status_bucket": recommendation.get("status"),
    "recommended_next_action": recommendation.get("recommended_next_action"),
    "report_output_file": str(REPORT_OUTPUT_FILE.resolve()),
}


{'overall_status': 'prompt_leakage_issue',
 'status_bucket': 'enum_instability',
 'recommended_next_action': 'Stabilize task-specific enum generation before the next smoke run.',
 'report_output_file': '/home/hyunlord/github/worldsim-training/outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z/analysis_report.json'}